# Unit 7: Materials Science — Hubbard Model via QPE

**Companion notebook for *What Quantum Computers Are Actually For***

We compute the ground-state energy of the 2-site Hubbard model, sweeping the
interaction strength $U/t$ to observe the Mott insulator transition. We run
a simplified QPE circuit on a cloud [Quokka](https://www.quokkacomputing.com/).

**What you'll see:**
1. The 2-site Hubbard model Hamiltonian
2. Exact diagonalisation (classical benchmark)
3. A QPE circuit for energy extraction
4. The metal-insulator transition as $U/t$ increases

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

import requests
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

QUOKKA = "quokka1"
QUOKKA_URL = f"http://{QUOKKA}.quokkacomputing.com/qsim/qasm"

def run_qasm(program: str, shots: int = 1024) -> dict:
    result = requests.post(QUOKKA_URL, json={"script": program, "count": shots}, verify=False)
    result.raise_for_status()
    return json.loads(result.content)

print(f"Connected to {QUOKKA_URL}")

## 1. The 2-site Hubbard model

$$H = -t \sum_{\sigma} (c_{1\sigma}^\dagger c_{2\sigma} + \text{h.c.}) + U \sum_i n_{i\uparrow} n_{i\downarrow}$$

2 sites, 2 electrons (half-filling), 4 spin-orbitals → 4 qubits (Jordan-Wigner).

In the 2-electron sector, the Hamiltonian has dimension 6. We can diagonalise it
exactly to get the benchmark.

In [ ]:
def hubbard_2site_energies(t_hop: float, U: float) -> np.ndarray:
    """Exact eigenvalues of the 2-site Hubbard model at half-filling.
    
    In the singlet sector, the Hamiltonian is 3×3:
    |1> = |↑↓, 0>  (both electrons on site 1)
    |2> = |0, ↑↓>  (both electrons on site 2)
    |3> = (|↑,↓> - |↓,↑>)/√2  (one on each, singlet)
    
    H = [[U, 0, -√2 t],
         [0, U, -√2 t],
         [-√2 t, -√2 t, 0]]
    """
    H = np.array([
        [U,    0,          -np.sqrt(2)*t_hop],
        [0,    U,          -np.sqrt(2)*t_hop],
        [-np.sqrt(2)*t_hop, -np.sqrt(2)*t_hop, 0]
    ])
    return np.sort(np.linalg.eigvalsh(H))


# Sweep U/t
t_hop = 1.0
U_values = np.linspace(0, 10, 50)
ground_energies = []

for U in U_values:
    energies = hubbard_2site_energies(t_hop, U)
    ground_energies.append(energies[0])

plt.figure(figsize=(8, 5))
plt.plot(U_values, ground_energies, '-', color='#009688', linewidth=2)
plt.xlabel('$U/t$')
plt.ylabel('Ground-state energy / $t$')
plt.title('2-site Hubbard model: Mott transition')
plt.axhline(y=0, color='gray', linestyle=':', alpha=0.3)
plt.annotate('Metal\n(delocalised)', xy=(1, -2.5), fontsize=11, color='#009688')
plt.annotate('Insulator\n(localised)', xy=(7, -0.5), fontsize=11, color='#FF5722')
plt.tight_layout()
plt.show()

print(f"At U/t=0: E₀ = {hubbard_2site_energies(1, 0)[0]:.4f}t")
print(f"At U/t=4: E₀ = {hubbard_2site_energies(1, 4)[0]:.4f}t")
print(f"At U/t=10: E₀ = {hubbard_2site_energies(1, 10)[0]:.4f}t")

## 2. QPE circuit for energy extraction

We use a simplified QPE circuit with 3 ancilla qubits to estimate the ground
state energy at a specific $U/t$ value. The key idea: QPE extracts the
eigenvalue $E$ as a phase $\phi = E \cdot t_{\text{evol}} / (2\pi)$.

In [ ]:
# Simplified QPE demonstration
# We encode the ground-state energy as a phase rotation
# and use QPE to read it out

# At U/t = 4, the exact ground state energy is:
U_target = 4.0
E_exact = hubbard_2site_energies(t_hop, U_target)[0]
print(f"Target: E₀ = {E_exact:.4f}t at U/t = {U_target}")

# Encode as a phase: φ = E * t_evol / (2π)
# Choose t_evol so the phase is representable with 3 bits
t_evol = 2 * np.pi / 8  # Makes φ = E/8
phase = E_exact * t_evol / (2 * np.pi)
# Wrap to [0, 1)
phase_wrapped = phase % 1

print(f"Phase φ = {phase_wrapped:.4f}")
print(f"Expected QPE output: k ≈ {phase_wrapped * 8:.1f} (out of 8)")

# Run the QPE circuit from Cookbook Recipe 10
# Using a pre-computed phase rotation as the "eigenvalue"
phase_angle = 2 * np.pi * phase_wrapped

qpe_circuit = f"""OPENQASM 2.0;
include "qelib1.inc";

qreg q[4];  // q[0-2] = ancilla (phase readout), q[3] = eigenstate
creg c[3];  // measure ancilla only

// Prepare eigenstate
x q[3];

// Hadamard on ancillas
h q[0];
h q[1];
h q[2];

// Controlled phase rotations (encoding E₀)
cp({phase_angle:.6f}) q[2], q[3];    // controlled-U^1
cp({2*phase_angle:.6f}) q[1], q[3];  // controlled-U^2
cp({4*phase_angle:.6f}) q[0], q[3];  // controlled-U^4

// Inverse QFT on ancillas
h q[0];
cp({-np.pi/2:.6f}) q[0], q[1];
h q[1];
cp({-np.pi/4:.6f}) q[0], q[2];
cp({-np.pi/2:.6f}) q[1], q[2];
h q[2];

// Bit reversal swap
cx q[0], q[2];
cx q[2], q[0];
cx q[0], q[2];

measure q[0] -> c[0];
measure q[1] -> c[1];
measure q[2] -> c[2];
"""

results = run_qasm(qpe_circuit, shots=1024)

print("\nQPE measurement results:")
for outcome in sorted(results.keys(), key=lambda x: results[x], reverse=True):
    k = int(outcome, 2)
    E_estimated = k / (8 * t_evol / (2 * np.pi))
    print(f"  {outcome} (k={k}): {results[outcome]:>4} counts → E ≈ {E_estimated:.3f}t")

# Best estimate
best_outcome = max(results, key=results.get)
k_best = int(best_outcome, 2)
E_qpe = k_best / (8 * t_evol / (2 * np.pi))
print(f"\nQPE estimate: E₀ ≈ {E_qpe:.4f}t")
print(f"Exact:        E₀ = {E_exact:.4f}t")
print(f"Error:        {abs(E_qpe - E_exact):.4f}t")

## What's next?

- **QFT deep dive:** the corresponding deep dive chapter
- **QPE deep dive:** the corresponding deep dive chapter
- Increase QPE precision by adding more ancilla qubits
- [Unit 8 — Climate & Energy](../manuscript/08-climate-energy.md): the full quantum simulation pipeline